### **Retriever pipeline from VectorStore**

In [2]:
from typing import List, Dict, Any

from utils import EmbeddingManager, VectorStore

In [7]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initializes the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieves relevant documents from a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for the query {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found.")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [8]:
rag_retriever = RAGRetriever(VectorStore(), EmbeddingManager())
rag_retriever

Vector store initialized. Collection: pdf_documents
Exisiting documents in collection: 47
Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1837.35it/s]


Model loaded successfully. Embedding dimension: 384


In [35]:
rag_retriever.retrieve("What is automation")

Retrieving documents for the query What is automation
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.09it/s]

Generated embeddings with shape (1, 384)
Retrieved 1 documents


[{'id': 'doc_e41d38e2_25',
  'content': '# **-** **Practical  5** **Aim: Understand the Automation Testing approach** **(Theory concept)**\n\nAutomation:\n\n\nAutomation is making a process automatic eliminating the need for\nhuman intervention. It is a self-controlling or self-moving process.\n\nAutomation Software offers automation wizards and commands of\n\nits own in addition to providing a task recording and re-play\ncapabilities. Using these programs, you can record an IT or business\n\ntask.\n\n\nBenefits of Automation\n\n\n    - Fast\n\n\n    - Reliable\n\n\n    Repeatable\n\n\n    Programmable\n\n\n    - Reusable\n\n\n    Makes Regression testing easy\n\n\n    Enables 24*78 Testing\n\n\n    - Robust verification.\n\n\n# **i**',
  'metadata': {'modDate': 'D:20260311171953Z',
   'creationdate': '',
   'trapped': '',
   'creator': '',
   'producer': 'Pdftools SDK',
   'author': '',
   'subject': '',
   'source': '..\\..\\data\\pdf\\STdisha5-6.pdf',
   'title': '',
   'creationDat